# Step 5: RAG with Microsoft Agent Framework

Same RAG pipeline as Step 4, but powered by [Microsoft Agent Framework](https://learn.microsoft.com/en-us/agent-framework/). The agent uses a function tool to perform hybrid search (keyword + vector) with semantic reranking on Azure AI Search, then generates answers with page citations automatically.

In [1]:
%pip install agent-framework agent-framework-openai azure-identity azure-search-documents python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from typing import Annotated

from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatCompletionClient
from azure.identity import DefaultAzureCredential as SyncDefaultAzureCredential, get_bearer_token_provider as sync_get_bearer_token_provider
from azure.identity.aio import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType
from openai import AzureOpenAI
from pydantic import Field
from dotenv import load_dotenv

load_dotenv(override=True)

# ── Async credential (for Agent Framework) ──
credential = DefaultAzureCredential()

# ── Sync credential (for OpenAI embeddings + Search) ──
sync_credential = SyncDefaultAzureCredential()
sync_token_provider = sync_get_bearer_token_provider(
    sync_credential, "https://cognitiveservices.azure.com/.default"
)

# ── Agent Framework chat client (Azure OpenAI) ──
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
chat_client = OpenAIChatCompletionClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=credential,
    model=CHAT_DEPLOYMENT,
)

# ── Embedding client (Azure OpenAI SDK — sync, for vectorising queries) ──
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
EMBEDDING_DIMS = 256  # Must match what was used in Step 3
embedding_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_ad_token_provider=sync_token_provider,
    api_version="2024-12-01-preview",
)

# ── Azure AI Search client (sync) ──
search_client = SearchClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-index"),
    credential=sync_credential,
)

print(f"Chat model: {CHAT_DEPLOYMENT}")
print(f"Embedding model: {EMBEDDING_DEPLOYMENT} (dims={EMBEDDING_DIMS})")
print(f"Search index: {os.getenv('AZURE_SEARCH_INDEX_NAME')}")
print("Ready!")

Chat model: gpt-4.1
Embedding model: text-embedding-3-large-460208 (dims=256)
Search index: rag-index
Ready!


## Define the search tool

The `@tool` decorator turns a plain Python function into a function tool that the agent can call whenever it needs to look up information from the Harry Potter books.

In [3]:
@tool(name="search_books", description="Search the Harry Potter books for information. Returns relevant passages with page numbers.")
def search_books(
    query: Annotated[str, Field(description="The search query to find relevant passages in the Harry Potter books.")],
    top_k: Annotated[int, Field(description="Number of results to return.")] = 5,
) -> str:
    """Hybrid search + semantic reranking over the Harry Potter book index."""
    # Embed the query
    response = embedding_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=[query],
        dimensions=EMBEDDING_DIMS,
    )
    query_vector = response.data[0].embedding

    # Hybrid search with semantic reranking
    results = search_client.search(
        search_text=query,
        vector_queries=[
            VectorizedQuery(vector=query_vector, k=top_k, fields="embedding")
        ],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default-semantic",
        top=top_k,
    )

    # Format results for the agent
    chunks = []
    for r in results:
        score_info = f"(reranker: {r.get('@search.reranker_score', 0):.2f})"
        chunks.append(f"[Page {r['page_number']}] {score_info}\n{r['content']}")

    if not chunks:
        return "No relevant passages found."

    return "\n\n---\n\n".join(chunks)

print("Search tool defined.")

Search tool defined.


## Create the Agent

The agent gets instructions about citing pages and the `search_books` tool. It will automatically decide when to call the tool based on the user's question.

In [4]:
agent = Agent(
    client=chat_client,
    name="HarryPotterRAG",
    instructions=(
        "You are a helpful assistant that answers questions about the Harry Potter books. "
        "Use the search_books tool to find relevant passages before answering. "
        "Always cite page numbers in your answer like (Page X). "
        "If the search results don't contain enough information, say so."
    ),
    tools=[search_books],
)

print(f"Agent '{agent.name}' created with tool: search_books")

Agent 'HarryPotterRAG' created with tool: search_books


## Ask questions

Use `agent.run()` for single-turn questions. The agent autonomously calls `search_books` and generates an answer with citations.

In [5]:
result = await agent.run(
    "What are the different names that the Dark Lord had in the book? Tell me in which parts of the book these names come up as well."
)
print(result.text)

ChatClientException: <class 'agent_framework_openai._chat_completion_client.OpenAIChatCompletionClient'> service failed to complete the prompt: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

In [27]:
result = await agent.run(
    "In Chamber of Secrets, how is Voldemort's name and how does he name himself?"
)
print(result.text)

In Chamber of Secrets, Voldemort reveals that his original name is Tom Marvolo Riddle. He then uses magic to rearrange the letters of his name to form the anagram "I am Lord Voldemort." He explains to Harry that he was already using this name at Hogwarts with his closest friends and did not want to keep his "filthy Muggle father's name," instead creating a new name that wizards would fear (Page 542).


In [28]:
result = await agent.run(
    "Who is Harry's godfather and how is he related to his parents?"
)
print(result.text)

Harry's godfather is Sirius Black. He was his parents' best friend, as Harry says: "He was my mum and dad’s best friend. He’s a convicted murderer, but he’s broken out of Wizard prison and he’s on the run. He likes to keep in touch with me, though..." (Page 939). In one of Lily Potter's letters, it is mentioned that "Sirius had bought him his first broomstick," further showing how close Sirius was to Harry's parents (Page 3127). So, Sirius Black is not only Harry’s godfather but also a close friend of both James and Lily Potter.


## Multi-turn conversation

Use a session to keep conversation history across turns so the agent can handle follow-up questions.

In [29]:
session = agent.create_session()

# Turn 1
response = await agent.run("What is the Philosopher's Stone?", session=session)
print(f"Turn 1:\n{response.text}\n")

# Turn 2 — follow-up referencing the previous answer
response = await agent.run("Who was trying to steal it and why?", session=session)
print(f"Turn 2:\n{response.text}\n")

Turn 1:
The Philosopher's Stone (also called the Sorcerer's Stone) is a legendary substance in the study of alchemy. It has astonishing powers: it can transform any metal into pure gold, and it produces the Elixir of Life, which makes the drinker immortal. In the story, there is only one known Stone in existence, owned by Nicolas Flamel, a famous alchemist (Page 198).

Turn 2:
It was Professor Quirrell who was trying to steal the Philosopher's Stone, but he was actually being controlled and possessed by Lord Voldemort. Voldemort wanted the Stone so that he could regain his body and become immortal by making the Elixir of Life that the Stone produces (Page 1494). Quirrell was following Voldemort's orders in attempting to steal the Stone from Hogwarts (Page 2438).

At first, Harry and his friends suspected Professor Snape, but it was later revealed that Snape was actually trying to protect the Stone, not steal it (Page 208).



## Streaming

Stream responses token-by-token for a chat-like experience.

In [30]:
print("Agent: ", end="", flush=True)
async for chunk in agent.run("Describe the Triwizard Tournament.", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

Agent: The Triwizard Tournament is a magical competition that was established around seven hundred years ago as a friendly contest between the three largest European wizarding schools: Hogwarts, Beauxbatons, and Durmstrang. Each school selects a champion to represent them, and these champions compete in three challenging magical tasks. The tasks are designed to test their magical ability, daring, powers of deduction, and ability to handle danger. The champion who performs best overall wins the Triwizard Cup (Page 1102, 1160).

Champions are chosen by an impartial selector called the Goblet of Fire, in which interested students place their names. The tournament is typically held at one of the three schools on a rotational basis (Page 1102, 1160).


## Cleanup

In [31]:
await credential.close()
print("Done!")

Done!
